In [0]:
%sql
-- =====================================================
-- DIAGNOSTIC INITIAL - Phase 7 Optimisation
-- Baseline AVANT toute optimisation
-- =====================================================

DESCRIBE DETAIL banking_lakehouse.gold.fact_transactions;

In [0]:
%sql
-- =====================================================
-- Diagnostic - silver.transactions
-- (table ayant subi plusieurs écritures : bug + correction)
-- =====================================================

DESCRIBE DETAIL banking_lakehouse.silver.transactions;

In [0]:
%sql
-- =====================================================
-- Historique complet des opérations Delta
-- (visualise notre incident + correction !)
-- =====================================================

DESCRIBE HISTORY banking_lakehouse.silver.transactions;

In [0]:
%sql
-- =====================================================
-- BENCHMARK AVANT optimisation
-- Requête typique : filtrer les transactions frauduleuses
-- =====================================================

SELECT COUNT(*), AVG(Amount)
FROM banking_lakehouse.gold.fact_transactions
WHERE transaction_type = 'Fraud'
AND hour_of_day BETWEEN 2 AND 7;

In [0]:
%sql
-- =====================================================
-- BENCHMARK AVANT - Requête filtrée par date
-- (cas d'usage typique justifiant le partitionnement)
-- =====================================================

SELECT COUNT(*), SUM(Amount)
FROM banking_lakehouse.gold.fact_transactions
WHERE transaction_date = '2026-01-01';

In [0]:
%sql
SELECT 
    COUNT(*) AS total_lignes,
    COUNT(DISTINCT transaction_id) AS ids_distincts,
    ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT transaction_id), 1) AS facteur_duplication
FROM banking_lakehouse.gold.fact_transactions;

In [0]:
%sql
DESCRIBE DETAIL banking_lakehouse.gold.fact_transactions;

In [0]:
%sql
SELECT COUNT(*), AVG(Amount)
FROM banking_lakehouse.gold.fact_transactions
WHERE transaction_type = 'Fraud'
AND hour_of_day BETWEEN 2 AND 7;

In [0]:
%sql
SELECT COUNT(*), SUM(Amount)
FROM banking_lakehouse.gold.fact_transactions
WHERE transaction_date = '2026-01-01';

In [0]:
%sql
-- =====================================================
-- OPTIMIZE + Z-ORDER sur fact_transactions
-- Réorganise physiquement les données pour accélérer
-- le data skipping sur nos colonnes de filtrage fréquentes
-- =====================================================

OPTIMIZE banking_lakehouse.gold.fact_transactions
ZORDER BY (transaction_date, transaction_type);

In [0]:
%sql
DESCRIBE DETAIL banking_lakehouse.gold.fact_transactions;

In [0]:
%sql
SELECT COUNT(*), SUM(Amount)
FROM banking_lakehouse.gold.fact_transactions
WHERE transaction_date = '2026-01-01';

In [0]:
%sql
-- Nouvelle baseline pour comparaison plus pertinente
SELECT COUNT(*), AVG(Amount)
FROM banking_lakehouse.gold.fact_transactions
WHERE hour_of_day = 3;

In [0]:
%sql
-- =====================================================
-- Partitionnement de fact_transactions par date
-- Démonstration du mécanisme (avec limite de cardinalité
-- honnêtement documentée : seulement 2 partitions ici)
-- =====================================================

CREATE OR REPLACE TABLE banking_lakehouse.gold.fact_transactions_partitioned
USING DELTA
PARTITIONED BY (transaction_date)
AS
SELECT * FROM banking_lakehouse.gold.fact_transactions;

In [0]:
%sql
DESCRIBE DETAIL banking_lakehouse.gold.fact_transactions_partitioned;

In [0]:
%sql
EXPLAIN FORMATTED
SELECT COUNT(*), SUM(Amount)
FROM banking_lakehouse.gold.fact_transactions_partitioned
WHERE transaction_date = '2026-01-01';

In [0]:
%sql
EXPLAIN FORMATTED
SELECT COUNT(*), SUM(Amount)
FROM banking_lakehouse.gold.fact_transactions
WHERE transaction_date = '2026-01-01';

In [0]:
%sql
-- Nettoyage : suppression de la table de démonstration
DROP TABLE IF EXISTS banking_lakehouse.gold.fact_transactions_partitioned;

In [0]:
%sql
-- =====================================================
-- VACUUM en mode DRY RUN - liste les fichiers qui SERAIENT supprimés
-- Sans rien supprimer réellement (sécurité)
-- =====================================================

VACUUM banking_lakehouse.silver.transactions DRY RUN;

In [0]:
%sql
-- =====================================================
-- VACUUM standard (respectant la politique de rétention)
-- Confirme que la commande est opérationnelle
-- =====================================================

VACUUM banking_lakehouse.gold.fact_transactions DRY RUN;